# Problem 8, Cartan structure equations (affine connection + local frame)

**Goal.** For an affine connection $\nabla$ and a local frame
$F = (X_a)$ (with dual coframe $e^a$), syntactically close the
local-component forms and their structure equations with the engine:

**(a)** Form-degree.
$$
\omega^a{}_b(\nabla)(V) := e^a(\nabla_V X_b),
\quad
Q_{ab}(\nabla, g)(V) := Q(\nabla, g)(V, X_a, X_b),
\quad
T^a(\nabla)(U,V) := e^a(T(\nabla)(U,V)),
\quad
R^a{}_b(\nabla)(U,V) := e^a(R(\nabla)(U,V)\,X_b)
$$
are defined. We will mechanically prove that $\omega$ and $Q$ are
$C^\infty$-linear in $V$ (i.e. 1-forms) and that $T$ and $R$ are
$C^\infty$-linear in both arguments and antisymmetric (i.e. 2-forms).

**(b)** Cartan I & II.
$$
\begin{aligned}
T^a &= d e^a + \omega^a{}_b \wedge e^b,\\
R^a{}_b &= d \omega^a{}_b + \omega^a{}_c \wedge \omega^c{}_b.
\end{aligned}
$$
We will reduce the difference of the two sides to $0$ by driving
engine + `simplify` to a fixed point.

## Setup

With a connection $\nabla$, a metric $g$ and a $\dim 3$ local frame $F$
we spin up two wrappers: `CartanFormPropertyProblem` for the form-degree
proofs, and `CartanStructureProblem` for Cartan I/II.

In [1]:
# Make jacopy importable when the notebook is opened directly.
try:
    import jacopy  # noqa: F401
except ModuleNotFoundError:
    import sys
    from pathlib import Path
    here = Path.cwd().resolve()
    for candidate in (here, *here.parents):
        if (candidate / "jacopy" / "__init__.py").is_file():
            sys.path.insert(0, str(candidate))
            break
    import jacopy  # noqa: F401

In [2]:
from jacopy.algebra.derivation import Derivation
from jacopy.calculus.connection import connection
from jacopy.calculus.local_frame import local_frame
from jacopy.calculus.metric import metric
from jacopy.core.expr import Symbol
from jacopy.display.jupyter import display_chain
from jacopy.library import CartanStructureProblem, CartanFormPropertyProblem
from jacopy.proof.chain import ProofChain

nabla = connection("∇")
g = metric("g")
F = local_frame("F", dim=3)

U = Derivation("U", 0)
V = Derivation("V", 0)
U1 = Derivation("U_1", 0)
U2 = Derivation("U_2", 0)
f = Symbol("f")

probA = CartanFormPropertyProblem(nabla, F, metric=g)
probB = CartanStructureProblem(nabla, F)
probA, probB

(CartanFormPropertyProblem(∇,F,g), CartanStructureProblem(∇,F))

## Part A, form-degree proofs

Problem 8 (a): $\omega^a{}_b$ and $Q_{ab}$ are 1-forms; $T^a$ and
$R^a{}_b$ are 2-forms. In every proof, LHS − RHS runs through the
engine and reduces to $0$. We show one representative property per
form below; in total `CartanFormPropertyProblem` exposes 14 `prove_*`
methods.

### A.1, $\omega^a{}_b$ 1-form: $\langle \omega^a{}_b, f\,V\rangle = f\,\langle\omega^a{}_b, V\rangle$

In [3]:
res_omega = probA.prove_omega_scalar_linear_in_V("a", "b", f, V)
print("OK?", res_omega.ok)
print("final:", res_omega.final._repr_inner())
print(f"engine steps: {len(res_omega.steps)}")
display_chain(ProofChain(res_omega.steps))

OK? True
final: 0
engine steps: 4


{\allowdisplaybreaks\scriptsize
\begin{gather*}
\langle \omega^a_b(∇),\, f \, V \rangle \to \langle e^a,\, ∇_{(f * V)(X_b)} \rangle \quad \text{[\ensuremath{\omega} definition [∇,F]]\,(axiom)} \\
∇_{(f * V)(X_b)} \to f \, ∇_{V(X_b)} \quad \text{[∇\_X X-scalar pull [∇]]\,(axiom)} \\
\langle e^a,\, f \, ∇_{V(X_b)} \rangle \to f \, \langle e^a,\, ∇_{V(X_b)} \rangle \quad \text{[Pairing C\ensuremath{\infty}-linearity: \ensuremath{\langle}\ensuremath{\alpha}, f\ensuremath{\cdot}X\ensuremath{\rangle} = f\ensuremath{\cdot}\ensuremath{\langle}\ensuremath{\alpha}, X\ensuremath{\rangle}]\,(axiom)} \\
\langle \omega^a_b(∇),\, V \rangle \to \langle e^a,\, ∇_{V(X_b)} \rangle \quad \text{[\ensuremath{\omega} definition [∇,F]]\,(axiom)}
\end{gather*}
}

In [4]:
for i, step in enumerate(ProofChain(res_omega.steps).steps, 1):
    tag = f"[{step.provenance_tag}]" if step.provenance_tag else ""
    print(f"[{i}] {step.rule} {tag}")
    print(f"    {step.before}")
    print(f" -> {step.after}")
    print()

[1] ω definition [∇,F] [axiom]
    ⟨ω^a_b(∇), (f * V)⟩
 -> ⟨e^a, ∇_(f * V)(X_b)⟩

[2] ∇_X X-scalar pull [∇] [axiom]
    ∇_(f * V)(X_b)
 -> (f * ∇_V(X_b))

[3] Pairing C∞-linearity: ⟨α, f·X⟩ = f·⟨α, X⟩ [axiom]
    ⟨e^a, (f * ∇_V(X_b))⟩
 -> (f * ⟨e^a, ∇_V(X_b)⟩)

[4] ω definition [∇,F] [axiom]
    ⟨ω^a_b(∇), V⟩
 -> ⟨e^a, ∇_V(X_b)⟩



### A.2, $Q_{ab}$ 1-form: $\langle Q_{ab}, f\,V\rangle = f\,\langle Q_{ab}, V\rangle$

In [5]:
res_Q = probA.prove_Q_scalar_linear_in_V("a", "b", f, V)
print("OK?", res_Q.ok)
print("final:", res_Q.final._repr_inner())
print(f"engine steps: {len(res_Q.steps)}")
display_chain(ProofChain(res_Q.steps))

OK? True
final: 0
engine steps: 3


{\allowdisplaybreaks\scriptsize
\begin{gather*}
\langle Q_{ab}(∇,g),\, f \, V \rangle \to f \, \langle Q_{ab}(∇,g),\, V \rangle \quad \text{[Pairing C\ensuremath{\infty}-linearity: \ensuremath{\langle}\ensuremath{\alpha}, f\ensuremath{\cdot}X\ensuremath{\rangle} = f\ensuremath{\cdot}\ensuremath{\langle}\ensuremath{\alpha}, X\ensuremath{\rangle}]\,(axiom)} \\
\langle Q_{ab}(∇,g),\, V \rangle \to Q(∇,g)(V,X_{a,X_b)} \quad \text{[Q form definition [∇,g,F]]\,(axiom)} \\
\langle Q_{ab}(∇,g),\, V \rangle \to Q(∇,g)(V,X_{a,X_b)} \quad \text{[Q form definition [∇,g,F]]\,(axiom)}
\end{gather*}
}

In [6]:
for i, step in enumerate(ProofChain(res_Q.steps).steps, 1):
    tag = f"[{step.provenance_tag}]" if step.provenance_tag else ""
    print(f"[{i}] {step.rule} {tag}")
    print(f"    {step.before}")
    print(f" -> {step.after}")
    print()

[1] Pairing C∞-linearity: ⟨α, f·X⟩ = f·⟨α, X⟩ [axiom]
    ⟨Q_ab(∇,g), (f * V)⟩
 -> (f * ⟨Q_ab(∇,g), V⟩)

[2] Q form definition [∇,g,F] [axiom]
    ⟨Q_ab(∇,g), V⟩
 -> Q(∇,g)(V,X_a,X_b)

[3] Q form definition [∇,g,F] [axiom]
    ⟨Q_ab(∇,g), V⟩
 -> Q(∇,g)(V,X_a,X_b)



### A.3, $T^a$ 2-form (antisymmetry): $T^a(U, V) + T^a(V, U) = 0$

In [7]:
res_T = probA.prove_T_antisymmetric("a", U, V)
print("OK?", res_T.ok)
print("final:", res_T.final._repr_inner())
print(f"engine steps: {len(res_T.steps)}")
display_chain(ProofChain(res_T.steps))

OK? True
final: 0
engine steps: 4


{\allowdisplaybreaks\scriptsize
\begin{gather*}
T^a(∇)\!\left(U,\, V\right) \to \langle e^a,\, T(∇)(U,V) \rangle \quad \text{[T form definition [∇,F]]\,(axiom)} \\
T^a(∇)\!\left(V,\, U\right) \to \langle e^a,\, T(∇)(V,U) \rangle \quad \text{[T form definition [∇,F]]\,(axiom)} \\
T(∇)(V,U) \to -T(∇)(U,V) \quad \text{[T antisymmetry [∇]]\,(axiom)} \\
\langle e^a,\, -T(∇)(U,V) \rangle \to -\langle e^a,\, T(∇)(U,V) \rangle \quad \text{[Pairing R-linearity]\,(axiom)}
\end{gather*}
}

In [8]:
for i, step in enumerate(ProofChain(res_T.steps).steps, 1):
    tag = f"[{step.provenance_tag}]" if step.provenance_tag else ""
    print(f"[{i}] {step.rule} {tag}")
    print(f"    {step.before}")
    print(f" -> {step.after}")
    print()

[1] T form definition [∇,F] [axiom]
    T^a(∇)(U, V)
 -> ⟨e^a, T(∇)(U,V)⟩

[2] T form definition [∇,F] [axiom]
    T^a(∇)(V, U)
 -> ⟨e^a, T(∇)(V,U)⟩

[3] T antisymmetry [∇] [axiom]
    T(∇)(V,U)
 -> (-T(∇)(U,V))

[4] Pairing R-linearity [axiom]
    ⟨e^a, (-T(∇)(U,V))⟩
 -> (-⟨e^a, T(∇)(U,V)⟩)



### A.3.b, $T^a$ scalar linearity in first slot: $T^a(f\,U, V) = f\,T^a(U, V)$

In [9]:
res_T_lin = probA.prove_T_scalar_linear_in_first("a", f, U, V)
print("OK?", res_T_lin.ok)
print("final:", res_T_lin.final._repr_inner())
print(f"engine steps: {len(res_T_lin.steps)}")
display_chain(ProofChain(res_T_lin.steps))

OK? True
final: 0
engine steps: 4


{\allowdisplaybreaks\scriptsize
\begin{gather*}
T^a(∇)\!\left(f \, U,\, V\right) \to \langle e^a,\, T(∇)((f * U),V) \rangle \quad \text{[T form definition [∇,F]]\,(axiom)} \\
T(∇)((f * U),V) \to f \, T(∇)(U,V) \quad \text{[T X-scalar pull [∇]]\,(axiom)} \\
\langle e^a,\, f \, T(∇)(U,V) \rangle \to f \, \langle e^a,\, T(∇)(U,V) \rangle \quad \text{[Pairing C\ensuremath{\infty}-linearity: \ensuremath{\langle}\ensuremath{\alpha}, f\ensuremath{\cdot}X\ensuremath{\rangle} = f\ensuremath{\cdot}\ensuremath{\langle}\ensuremath{\alpha}, X\ensuremath{\rangle}]\,(axiom)} \\
T^a(∇)\!\left(U,\, V\right) \to \langle e^a,\, T(∇)(U,V) \rangle \quad \text{[T form definition [∇,F]]\,(axiom)}
\end{gather*}
}

In [10]:
for i, step in enumerate(ProofChain(res_T_lin.steps).steps, 1):
    tag = f"[{step.provenance_tag}]" if step.provenance_tag else ""
    print(f"[{i}] {step.rule} {tag}")
    print(f"    {step.before}")
    print(f" -> {step.after}")
    print()

[1] T form definition [∇,F] [axiom]
    T^a(∇)((f * U), V)
 -> ⟨e^a, T(∇)((f * U),V)⟩

[2] T X-scalar pull [∇] [axiom]
    T(∇)((f * U),V)
 -> (f * T(∇)(U,V))

[3] Pairing C∞-linearity: ⟨α, f·X⟩ = f·⟨α, X⟩ [axiom]
    ⟨e^a, (f * T(∇)(U,V))⟩
 -> (f * ⟨e^a, T(∇)(U,V)⟩)

[4] T form definition [∇,F] [axiom]
    T^a(∇)(U, V)
 -> ⟨e^a, T(∇)(U,V)⟩



### A.3.c, $T^a$ additivity in first slot: $T^a(U_1 + U_2, V) = T^a(U_1, V) + T^a(U_2, V)$

In [11]:
res_T_add = probA.prove_T_additive_in_first("a", U1, U2, V)
print("OK?", res_T_add.ok)
print("final:", res_T_add.final._repr_inner())
print(f"engine steps: {len(res_T_add.steps)}")
display_chain(ProofChain(res_T_add.steps))

OK? True
final: 0
engine steps: 5


{\allowdisplaybreaks\scriptsize
\begin{gather*}
T^a(∇)\!\left(U_1 + U_2,\, V\right) \to \langle e^a,\, T(∇)((U_{1 + U_2),V)} \rangle \quad \text{[T form definition [∇,F]]\,(axiom)} \\
T(∇)((U_{1 + U_2),V)} \to T(∇)(U_1,V) + T(∇)(U_2,V) \quad \text{[T X-linearity [∇]]\,(axiom)} \\
\langle e^a,\, T(∇)(U_1,V) + T(∇)(U_2,V) \rangle \to \langle e^a,\, T(∇)(U_1,V) \rangle + \langle e^a,\, T(∇)(U_2,V) \rangle \quad \text{[Pairing R-linearity]\,(axiom)} \\
T^a(∇)\!\left(U_1,\, V\right) \to \langle e^a,\, T(∇)(U_1,V) \rangle \quad \text{[T form definition [∇,F]]\,(axiom)} \\
T^a(∇)\!\left(U_2,\, V\right) \to \langle e^a,\, T(∇)(U_2,V) \rangle \quad \text{[T form definition [∇,F]]\,(axiom)}
\end{gather*}
}

In [12]:
for i, step in enumerate(ProofChain(res_T_add.steps).steps, 1):
    tag = f"[{step.provenance_tag}]" if step.provenance_tag else ""
    print(f"[{i}] {step.rule} {tag}")
    print(f"    {step.before}")
    print(f" -> {step.after}")
    print()

[1] T form definition [∇,F] [axiom]
    T^a(∇)((U_1 + U_2), V)
 -> ⟨e^a, T(∇)((U_1 + U_2),V)⟩

[2] T X-linearity [∇] [axiom]
    T(∇)((U_1 + U_2),V)
 -> (T(∇)(U_1,V) + T(∇)(U_2,V))

[3] Pairing R-linearity [axiom]
    ⟨e^a, (T(∇)(U_1,V) + T(∇)(U_2,V))⟩
 -> (⟨e^a, T(∇)(U_1,V)⟩ + ⟨e^a, T(∇)(U_2,V)⟩)

[4] T form definition [∇,F] [axiom]
    T^a(∇)(U_1, V)
 -> ⟨e^a, T(∇)(U_1,V)⟩

[5] T form definition [∇,F] [axiom]
    T^a(∇)(U_2, V)
 -> ⟨e^a, T(∇)(U_2,V)⟩



**A.3 sonuç**: $T^a$ slot 1 üzerinde C∞-linear (scalar + additive) ve slot pair üzerinde antisymmetric. Antisymmetry sayesinde slot 2 linearity otomatik:

$$T^a(U, fV) = -T^a(fV, U) = -f\,T^a(V, U) = f\,T^a(U, V).$$

Yani üç madde birlikte: $T^a$ **C∞-bilinear + alternating** ⇔ $T^a$ vector-valued 2-form.

### A.4, $R^a{}_b$ 2-form (antisymmetry): $R^a{}_b(U, V) + R^a{}_b(V, U) = 0$

In [13]:
res_R = probA.prove_R_antisymmetric("a", "b", U, V)
print("OK?", res_R.ok)
print("final:", res_R.final._repr_inner())
print(f"engine steps: {len(res_R.steps)}")
display_chain(ProofChain(res_R.steps))

OK? True
final: 0
engine steps: 4


{\allowdisplaybreaks\scriptsize
\begin{gather*}
R^a_b(∇)\!\left(U,\, V\right) \to \langle e^a,\, R(∇)(U,V)(X_b) \rangle \quad \text{[R form definition [∇,F]]\,(axiom)} \\
R^a_b(∇)\!\left(V,\, U\right) \to \langle e^a,\, R(∇)(V,U)(X_b) \rangle \quad \text{[R form definition [∇,F]]\,(axiom)} \\
R(∇)(V,U)(X_b) \to -R(∇)(U,V)(X_b) \quad \text{[R XY-antisymmetry [∇]]\,(axiom)} \\
\langle e^a,\, -R(∇)(U,V)(X_b) \rangle \to -\langle e^a,\, R(∇)(U,V)(X_b) \rangle \quad \text{[Pairing R-linearity]\,(axiom)}
\end{gather*}
}

In [14]:
for i, step in enumerate(ProofChain(res_R.steps).steps, 1):
    tag = f"[{step.provenance_tag}]" if step.provenance_tag else ""
    print(f"[{i}] {step.rule} {tag}")
    print(f"    {step.before}")
    print(f" -> {step.after}")
    print()

[1] R form definition [∇,F] [axiom]
    R^a_b(∇)(U, V)
 -> ⟨e^a, R(∇)(U,V)(X_b)⟩

[2] R form definition [∇,F] [axiom]
    R^a_b(∇)(V, U)
 -> ⟨e^a, R(∇)(V,U)(X_b)⟩

[3] R XY-antisymmetry [∇] [axiom]
    R(∇)(V,U)(X_b)
 -> (-R(∇)(U,V)(X_b))

[4] Pairing R-linearity [axiom]
    ⟨e^a, (-R(∇)(U,V)(X_b))⟩
 -> (-⟨e^a, R(∇)(U,V)(X_b)⟩)



### A.4.b, $R^a{}_b$ scalar linearity in first slot: $R^a{}_b(f\,U, V) = f\,R^a{}_b(U, V)$

In [15]:
res_R_lin = probA.prove_R_scalar_linear_in_first("a", "b", f, U, V)
print("OK?", res_R_lin.ok)
print("final:", res_R_lin.final._repr_inner())
print(f"engine steps: {len(res_R_lin.steps)}")
display_chain(ProofChain(res_R_lin.steps))

OK? True
final: 0
engine steps: 4


{\allowdisplaybreaks\scriptsize
\begin{gather*}
R^a_b(∇)\!\left(f \, U,\, V\right) \to \langle e^a,\, R(∇)((f * U),V)(X_b) \rangle \quad \text{[R form definition [∇,F]]\,(axiom)} \\
R(∇)((f * U),V)(X_b) \to f \, R(∇)(U,V)(X_b) \quad \text{[R X-scalar pull [∇]]\,(axiom)} \\
\langle e^a,\, f \, R(∇)(U,V)(X_b) \rangle \to f \, \langle e^a,\, R(∇)(U,V)(X_b) \rangle \quad \text{[Pairing C\ensuremath{\infty}-linearity: \ensuremath{\langle}\ensuremath{\alpha}, f\ensuremath{\cdot}X\ensuremath{\rangle} = f\ensuremath{\cdot}\ensuremath{\langle}\ensuremath{\alpha}, X\ensuremath{\rangle}]\,(axiom)} \\
R^a_b(∇)\!\left(U,\, V\right) \to \langle e^a,\, R(∇)(U,V)(X_b) \rangle \quad \text{[R form definition [∇,F]]\,(axiom)}
\end{gather*}
}

In [16]:
for i, step in enumerate(ProofChain(res_R_lin.steps).steps, 1):
    tag = f"[{step.provenance_tag}]" if step.provenance_tag else ""
    print(f"[{i}] {step.rule} {tag}")
    print(f"    {step.before}")
    print(f" -> {step.after}")
    print()

[1] R form definition [∇,F] [axiom]
    R^a_b(∇)((f * U), V)
 -> ⟨e^a, R(∇)((f * U),V)(X_b)⟩

[2] R X-scalar pull [∇] [axiom]
    R(∇)((f * U),V)(X_b)
 -> (f * R(∇)(U,V)(X_b))

[3] Pairing C∞-linearity: ⟨α, f·X⟩ = f·⟨α, X⟩ [axiom]
    ⟨e^a, (f * R(∇)(U,V)(X_b))⟩
 -> (f * ⟨e^a, R(∇)(U,V)(X_b)⟩)

[4] R form definition [∇,F] [axiom]
    R^a_b(∇)(U, V)
 -> ⟨e^a, R(∇)(U,V)(X_b)⟩



### A.4.c, $R^a{}_b$ additivity in first slot: $R^a{}_b(U_1 + U_2, V) = R^a{}_b(U_1, V) + R^a{}_b(U_2, V)$

In [17]:
res_R_add = probA.prove_R_additive_in_first("a", "b", U1, U2, V)
print("OK?", res_R_add.ok)
print("final:", res_R_add.final._repr_inner())
print(f"engine steps: {len(res_R_add.steps)}")
display_chain(ProofChain(res_R_add.steps))

OK? True
final: 0
engine steps: 5


{\allowdisplaybreaks\scriptsize
\begin{gather*}
R^a_b(∇)\!\left(U_1 + U_2,\, V\right) \to \langle e^a,\, R(∇)((U_{1 + U_{2),V)(X_b)}} \rangle \quad \text{[R form definition [∇,F]]\,(axiom)} \\
R(∇)((U_{1 + U_{2),V)(X_b)}} \to R(∇)(U_{1,V)(X_b)} + R(∇)(U_{2,V)(X_b)} \quad \text{[R X-linearity [∇]]\,(axiom)} \\
\langle e^a,\, R(∇)(U_{1,V)(X_b)} + R(∇)(U_{2,V)(X_b)} \rangle \to \langle e^a,\, R(∇)(U_{1,V)(X_b)} \rangle + \langle e^a,\, R(∇)(U_{2,V)(X_b)} \rangle \quad \text{[Pairing R-linearity]\,(axiom)} \\
R^a_b(∇)\!\left(U_1,\, V\right) \to \langle e^a,\, R(∇)(U_{1,V)(X_b)} \rangle \quad \text{[R form definition [∇,F]]\,(axiom)} \\
R^a_b(∇)\!\left(U_2,\, V\right) \to \langle e^a,\, R(∇)(U_{2,V)(X_b)} \rangle \quad \text{[R form definition [∇,F]]\,(axiom)}
\end{gather*}
}

In [18]:
for i, step in enumerate(ProofChain(res_R_add.steps).steps, 1):
    tag = f"[{step.provenance_tag}]" if step.provenance_tag else ""
    print(f"[{i}] {step.rule} {tag}")
    print(f"    {step.before}")
    print(f" -> {step.after}")
    print()

[1] R form definition [∇,F] [axiom]
    R^a_b(∇)((U_1 + U_2), V)
 -> ⟨e^a, R(∇)((U_1 + U_2),V)(X_b)⟩

[2] R X-linearity [∇] [axiom]
    R(∇)((U_1 + U_2),V)(X_b)
 -> (R(∇)(U_1,V)(X_b) + R(∇)(U_2,V)(X_b))

[3] Pairing R-linearity [axiom]
    ⟨e^a, (R(∇)(U_1,V)(X_b) + R(∇)(U_2,V)(X_b))⟩
 -> (⟨e^a, R(∇)(U_1,V)(X_b)⟩ + ⟨e^a, R(∇)(U_2,V)(X_b)⟩)

[4] R form definition [∇,F] [axiom]
    R^a_b(∇)(U_1, V)
 -> ⟨e^a, R(∇)(U_1,V)(X_b)⟩

[5] R form definition [∇,F] [axiom]
    R^a_b(∇)(U_2, V)
 -> ⟨e^a, R(∇)(U_2,V)(X_b)⟩



**A.4 sonuç**: $R^a{}_b$ slot 1 üzerinde C∞-linear, slot pair üzerinde antisymmetric ⇒ slot 2 linearity de otomatik (yukarıdaki argüman). Yani $R^a{}_b$ **endomorphism-valued 2-form**.

## Part B.1, Cartan I structure equation

$$
T^a(U, V) \;=\; (d e^a)(U, V) \;+\; \sum_b (\omega^a{}_b \wedge e^b)(U, V).
$$

The left side starts from the `T^a` form and unfolds into
$\nabla_U V - \nabla_V U - [U,V]_{VF}$; the right side expands via the
Cartan-Koszul intrinsic definition for `de^a` plus the sum-over-$b$
wedge alternation. When both sides reduce to identical
`Pairing`/`Act` atoms the difference drops to $0$.

In [19]:
lhs_I = probB.first_cartan_lhs(U, V, "a")
rhs_I = probB.first_cartan_rhs(U, V, "a")
print("Cartan I LHS:", lhs_I._repr_inner())
print("Cartan I RHS:", rhs_I._repr_inner())

Cartan I LHS: T^a(∇)(U, V)
Cartan I RHS: (d(e^a)(U, V) + Σ_b̂∈F (ω^a_b̂(∇) ∧ e^b̂)(U, V))


In [20]:
res_I = probB.prove_first_cartan(U, V, "a")
chain_I = ProofChain(res_I.steps)
print("OK?", res_I.ok)
print("final:", res_I.final._repr_inner())
print(f"engine steps: {len(chain_I)}")
display_chain(chain_I)

OK? True
final: 0
engine steps: 49


{\allowdisplaybreaks\scriptsize
\begin{gather*}
T^a(∇)\!\left(U,\, V\right) \to \langle e^a,\, T(∇)(U,V) \rangle \quad \text{[T form definition [∇,F]]\,(axiom)} \\
T(∇)(U,V) \to ∇_U(V) - ∇_V(U) - [U,V]_{VF} \quad \text{[T(∇)(X,Y) = ∇\_X Y \ensuremath{-} ∇\_Y X \ensuremath{-} [X,Y]\_VF [∇]]\,(axiom)} \\
∇_U(V) \to ∇_{U(\Sigma_{â∈F (\langlee^â, V\rangle * X_â))}} \quad \text{[∇\_X Y \ensuremath{\to} ∇\_X (\ensuremath{\Sigma}\_a e^a(Y)\ensuremath{\cdot}X\_a) [∇,F]]\,(axiom)} \\
∇_{U(\Sigma_{â∈F (\langlee^â, V\rangle * X_â))}} \to \Sigma_{â∈F ∇_{U((\langlee^â, V\rangle * X_â))}} \quad \text{[∇\_X \ensuremath{\Sigma}\_d body \ensuremath{\to} \ensuremath{\Sigma}\_d ∇\_X body [∇]]\,(axiom)} \\
∇_{U((\langlee^â, V\rangle * X_â))} \to U\!\left(\langle e^â,\, V \rangle\right) \, X_a + \langle e^â,\, V \rangle \, ∇_{U(X_â)} \quad \text{[∇\_X Y-Leibniz [∇]]\,(axiom)} \\
∇_{U(X_â)} \to \Sigma_{ĉ∈F (\langle\omega^ĉ_{â(∇), U\rangle * X_ĉ)}} \quad \text{[∇\_V X\_b \ensuremath{\to} \ensuremath{\Sigma}\_c \ensuremath{\omega}^c\_b(V)\ensuremath{\cdot}X\_c [∇,F]]\,(axiom)} \\
\Sigma_{â∈F ((U(\langlee^â, V\rangle) * X_{â) + (\langlee^â, V\rangle * \Sigma_{ĉ∈F (\langle\omega^ĉ_{â(∇), U\rangle * X_ĉ)))}}}} \to \Sigma_{â∈F (U(\langlee^â, V\rangle) * X_â)} + \Sigma_{â∈F (\langlee^â, V\rangle * \Sigma_{ĉ∈F (\langle\omega^ĉ_{â(∇), U\rangle * X_ĉ))}}} \quad \text{[\ensuremath{\Sigma} over Sum: distribute]\,(axiom)} \\
∇_V(U) \to ∇_{V(\Sigma_{â∈F (\langlee^â, U\rangle * X_â))}} \quad \text{[∇\_X Y \ensuremath{\to} ∇\_X (\ensuremath{\Sigma}\_a e^a(Y)\ensuremath{\cdot}X\_a) [∇,F]]\,(axiom)} \\
∇_{V(\Sigma_{â∈F (\langlee^â, U\rangle * X_â))}} \to \Sigma_{â∈F ∇_{V((\langlee^â, U\rangle * X_â))}} \quad \text{[∇\_X \ensuremath{\Sigma}\_d body \ensuremath{\to} \ensuremath{\Sigma}\_d ∇\_X body [∇]]\,(axiom)} \\
∇_{V((\langlee^â, U\rangle * X_â))} \to V\!\left(\langle e^â,\, U \rangle\right) \, X_a + \langle e^â,\, U \rangle \, ∇_{V(X_â)} \quad \text{[∇\_X Y-Leibniz [∇]]\,(axiom)} \\
∇_{V(X_â)} \to \Sigma_{ĉ∈F (\langle\omega^ĉ_{â(∇), V\rangle * X_ĉ)}} \quad \text{[∇\_V X\_b \ensuremath{\to} \ensuremath{\Sigma}\_c \ensuremath{\omega}^c\_b(V)\ensuremath{\cdot}X\_c [∇,F]]\,(axiom)} \\
\Sigma_{â∈F ((V(\langlee^â, U\rangle) * X_{â) + (\langlee^â, U\rangle * \Sigma_{ĉ∈F (\langle\omega^ĉ_{â(∇), V\rangle * X_ĉ)))}}}} \to \Sigma_{â∈F (V(\langlee^â, U\rangle) * X_â)} + \Sigma_{â∈F (\langlee^â, U\rangle * \Sigma_{ĉ∈F (\langle\omega^ĉ_{â(∇), V\rangle * X_ĉ))}}} \quad \text{[\ensuremath{\Sigma} over Sum: distribute]\,(axiom)} \\
\langle e^a,\, \left(\Sigma_{â∈F (U(\langlee^â, V\rangle) * X_â)} + \Sigma_{â∈F (\langlee^â, V\rangle * \Sigma_{ĉ∈F (\langle\omega^ĉ_{â(∇), U\rangle * X_ĉ))}}}\right) - \left(\Sigma_{â∈F (V(\langlee^â, U\rangle) * X_â)} + \Sigma_{â∈F (\langlee^â, U\rangle * \Sigma_{ĉ∈F (\langle\omega^ĉ_{â(∇), V\rangle * X_ĉ))}}}\right) - [U,V]_{VF} \rangle \to \langle e^a,\, \Sigma_{â∈F (U(\langlee^â, V\rangle) * X_â)} + \Sigma_{â∈F (\langlee^â, V\rangle * \Sigma_{ĉ∈F (\langle\omega^ĉ_{â(∇), U\rangle * X_ĉ))}}} \rangle + \langle e^a,\, -\left(\Sigma_{â∈F (V(\langlee^â, U\rangle) * X_â)} + \Sigma_{â∈F (\langlee^â, U\rangle * \Sigma_{ĉ∈F (\langle\omega^ĉ_{â(∇), V\rangle * X_ĉ))}}}\right) \rangle + \langle e^a,\, -[U,V]_{VF} \rangle \quad \text{[Pairing R-linearity]\,(axiom)} \\
\langle e^a,\, \Sigma_{â∈F (U(\langlee^â, V\rangle) * X_â)} + \Sigma_{â∈F (\langlee^â, V\rangle * \Sigma_{ĉ∈F (\langle\omega^ĉ_{â(∇), U\rangle * X_ĉ))}}} \rangle \to \langle e^a,\, \Sigma_{â∈F (U(\langlee^â, V\rangle) * X_â)} \rangle + \langle e^a,\, \Sigma_{â∈F (\langlee^â, V\rangle * \Sigma_{ĉ∈F (\langle\omega^ĉ_{â(∇), U\rangle * X_ĉ))}}} \rangle \quad \text{[Pairing R-linearity]\,(axiom)} \\
\langle e^a,\, \Sigma_{â∈F (U(\langlee^â, V\rangle) * X_â)} \rangle \to \Sigma_{â∈F \langlee^a, (U(\langlee^â, V\rangle) * X_â)\rangle} \quad \text{[\ensuremath{\langle}\ensure

In [21]:
for i, step in enumerate(chain_I.steps, 1):
    tag = f"[{step.provenance_tag}]" if step.provenance_tag else ""
    print(f"[{i}] {step.rule} {tag}")
    print(f"    {step.before}")
    print(f" -> {step.after}")
    print()

[1] T form definition [∇,F] [axiom]
    T^a(∇)(U, V)
 -> ⟨e^a, T(∇)(U,V)⟩

[2] T(∇)(X,Y) = ∇_X Y − ∇_Y X − [X,Y]_VF [∇] [axiom]
    T(∇)(U,V)
 -> (∇_U(V) + (-∇_V(U)) + (-[U,V]_VF))

[3] ∇_X Y → ∇_X (Σ_a e^a(Y)·X_a) [∇,F] [axiom]
    ∇_U(V)
 -> ∇_U(Σ_â∈F (⟨e^â, V⟩ * X_â))

[4] ∇_X Σ_d body → Σ_d ∇_X body [∇] [axiom]
    ∇_U(Σ_â∈F (⟨e^â, V⟩ * X_â))
 -> Σ_â∈F ∇_U((⟨e^â, V⟩ * X_â))

[5] ∇_X Y-Leibniz [∇] [axiom]
    ∇_U((⟨e^â, V⟩ * X_â))
 -> ((U(⟨e^â, V⟩) * X_â) + (⟨e^â, V⟩ * ∇_U(X_â)))

[6] ∇_V X_b → Σ_c ω^c_b(V)·X_c [∇,F] [axiom]
    ∇_U(X_â)
 -> Σ_ĉ∈F (⟨ω^ĉ_â(∇), U⟩ * X_ĉ)

[7] Σ over Sum: distribute [axiom]
    Σ_â∈F ((U(⟨e^â, V⟩) * X_â) + (⟨e^â, V⟩ * Σ_ĉ∈F (⟨ω^ĉ_â(∇), U⟩ * X_ĉ)))
 -> (Σ_â∈F (U(⟨e^â, V⟩) * X_â) + Σ_â∈F (⟨e^â, V⟩ * Σ_ĉ∈F (⟨ω^ĉ_â(∇), U⟩ * X_ĉ)))

[8] ∇_X Y → ∇_X (Σ_a e^a(Y)·X_a) [∇,F] [axiom]
    ∇_V(U)
 -> ∇_V(Σ_â∈F (⟨e^â, U⟩ * X_â))

[9] ∇_X Σ_d body → Σ_d ∇_X body [∇] [axiom]
    ∇_V(Σ_â∈F (⟨e^â, U⟩ * X_â))
 -> Σ_â∈F ∇_V((⟨e

## Part B.2, Cartan II structure equation

$$
R^a{}_b(U, V) \;=\; (d \omega^a{}_b)(U, V) \;+\; \sum_c (\omega^a{}_c \wedge \omega^c{}_b)(U, V).
$$

Same wrapper, same 24-rule engine bundle; the only difference: the
left side starts from the $R^a{}_b$ form and unfolds into
$\nabla_U \nabla_V X_b - \nabla_V \nabla_U X_b - \nabla_{[U,V]_{VF}} X_b$,
while the right side expands $d\omega^a{}_b$ via Cartan-Koszul plus a
second $\omega$ wedge.

In [22]:
lhs_II = probB.second_cartan_lhs(U, V, "a", "b")
rhs_II = probB.second_cartan_rhs(U, V, "a", "b")
print("Cartan II LHS:", lhs_II._repr_inner())
print("Cartan II RHS:", rhs_II._repr_inner())

Cartan II LHS: R^a_b(∇)(U, V)
Cartan II RHS: (d(ω^a_b(∇))(U, V) + Σ_ĉ∈F (ω^a_ĉ(∇) ∧ ω^ĉ_b(∇))(U, V))


In [23]:
res_II = probB.prove_second_cartan(U, V, "a", "b")
chain_II = ProofChain(res_II.steps)
print("OK?", res_II.ok)
print("final:", res_II.final._repr_inner())
print(f"engine steps: {len(chain_II)}")
display_chain(chain_II)

OK? True
final: 0
engine steps: 54


{\allowdisplaybreaks\scriptsize
\begin{gather*}
R^a_b(∇)\!\left(U,\, V\right) \to \langle e^a,\, R(∇)(U,V)(X_b) \rangle \quad \text{[R form definition [∇,F]]\,(axiom)} \\
R(∇)(U,V)(X_b) \to ∇_{U(∇_{V(X_b))}} - ∇_{V(∇_{U(X_b))}} - ∇_{[U,V]_{VF(X_b)}} \quad \text{[R(∇)(X,Y)Z = ∇\_X ∇\_Y Z \ensuremath{-} ∇\_Y ∇\_X Z \ensuremath{-} ∇\_[X,Y]\_VF Z [∇]]\,(axiom)} \\
∇_{V(X_b)} \to \Sigma_{ĉ∈F (\langle\omega^ĉ_{b(∇), V\rangle * X_ĉ)}} \quad \text{[∇\_V X\_b \ensuremath{\to} \ensuremath{\Sigma}\_c \ensuremath{\omega}^c\_b(V)\ensuremath{\cdot}X\_c [∇,F]]\,(axiom)} \\
∇_{U(\Sigma_{ĉ∈F (\langle\omega^ĉ_{b(∇), V\rangle * X_ĉ))}}} \to \Sigma_{ĉ∈F ∇_{U((\langle\omega^ĉ_{b(∇), V\rangle * X_ĉ))}}} \quad \text{[∇\_X \ensuremath{\Sigma}\_d body \ensuremath{\to} \ensuremath{\Sigma}\_d ∇\_X body [∇]]\,(axiom)} \\
∇_{U((\langle\omega^ĉ_{b(∇), V\rangle * X_ĉ))}} \to U\!\left(\langle \omega^ĉ_b(∇),\, V \rangle\right) \, X_c + \langle \omega^ĉ_b(∇),\, V \rangle \, ∇_{U(X_ĉ)} \quad \text{[∇\_X Y-Leibniz [∇]]\,(axiom)} \\
∇_{U(X_ĉ)} \to \Sigma_{c1̂∈F (\langle\omega^c1̂_{ĉ(∇), U\rangle * X_{c1}̂)}} \quad \text{[∇\_V X\_b \ensuremath{\to} \ensuremath{\Sigma}\_c \ensuremath{\omega}^c\_b(V)\ensuremath{\cdot}X\_c [∇,F]]\,(axiom)} \\
\Sigma_{ĉ∈F ((U(\langle\omega^ĉ_{b(∇), V\rangle) * X_{ĉ) + (\langle\omega^ĉ_{b(∇), V\rangle * \Sigma_{c1̂∈F (\langle\omega^c1̂_{ĉ(∇), U\rangle * X_{c1}̂)))}}}}}} \to \Sigma_{ĉ∈F (U(\langle\omega^ĉ_{b(∇), V\rangle) * X_ĉ)}} + \Sigma_{ĉ∈F (\langle\omega^ĉ_{b(∇), V\rangle * \Sigma_{c1̂∈F (\langle\omega^c1̂_{ĉ(∇), U\rangle * X_{c1}̂))}}}} \quad \text{[\ensuremath{\Sigma} over Sum: distribute]\,(axiom)} \\
∇_{U(X_b)} \to \Sigma_{ĉ∈F (\langle\omega^ĉ_{b(∇), U\rangle * X_ĉ)}} \quad \text{[∇\_V X\_b \ensuremath{\to} \ensuremath{\Sigma}\_c \ensuremath{\omega}^c\_b(V)\ensuremath{\cdot}X\_c [∇,F]]\,(axiom)} \\
∇_{V(\Sigma_{ĉ∈F (\langle\omega^ĉ_{b(∇), U\rangle * X_ĉ))}}} \to \Sigma_{ĉ∈F ∇_{V((\langle\omega^ĉ_{b(∇), U\rangle * X_ĉ))}}} \quad \text{[∇\_X \ensuremath{\Sigma}\_d body \ensuremath{\to} \ensuremath{\Sigma}\_d ∇\_X body [∇]]\,(axiom)} \\
∇_{V((\langle\omega^ĉ_{b(∇), U\rangle * X_ĉ))}} \to V\!\left(\langle \omega^ĉ_b(∇),\, U \rangle\right) \, X_c + \langle \omega^ĉ_b(∇),\, U \rangle \, ∇_{V(X_ĉ)} \quad \text{[∇\_X Y-Leibniz [∇]]\,(axiom)} \\
∇_{V(X_ĉ)} \to \Sigma_{c1̂∈F (\langle\omega^c1̂_{ĉ(∇), V\rangle * X_{c1}̂)}} \quad \text{[∇\_V X\_b \ensuremath{\to} \ensuremath{\Sigma}\_c \ensuremath{\omega}^c\_b(V)\ensuremath{\cdot}X\_c [∇,F]]\,(axiom)} \\
\Sigma_{ĉ∈F ((V(\langle\omega^ĉ_{b(∇), U\rangle) * X_{ĉ) + (\langle\omega^ĉ_{b(∇), U\rangle * \Sigma_{c1̂∈F (\langle\omega^c1̂_{ĉ(∇), V\rangle * X_{c1}̂)))}}}}}} \to \Sigma_{ĉ∈F (V(\langle\omega^ĉ_{b(∇), U\rangle) * X_ĉ)}} + \Sigma_{ĉ∈F (\langle\omega^ĉ_{b(∇), U\rangle * \Sigma_{c1̂∈F (\langle\omega^c1̂_{ĉ(∇), V\rangle * X_{c1}̂))}}}} \quad \text{[\ensuremath{\Sigma} over Sum: distribute]\,(axiom)} \\
∇_{[U,V]_{VF(X_b)}} \to \Sigma_{ĉ∈F (\langle\omega^ĉ_{b(∇), [U,V]_{VF\rangle * X_ĉ)}}} \quad \text{[∇\_V X\_b \ensuremath{\to} \ensuremath{\Sigma}\_c \ensuremath{\omega}^c\_b(V)\ensuremath{\cdot}X\_c [∇,F]]\,(axiom)} \\
\langle e^a,\, \left(\Sigma_{ĉ∈F (U(\langle\omega^ĉ_{b(∇), V\rangle) * X_ĉ)}} + \Sigma_{ĉ∈F (\langle\omega^ĉ_{b(∇), V\rangle * \Sigma_{c1̂∈F (\langle\omega^c1̂_{ĉ(∇), U\rangle * X_{c1}̂))}}}}\right) - \left(\Sigma_{ĉ∈F (V(\langle\omega^ĉ_{b(∇), U\rangle) * X_ĉ)}} + \Sigma_{ĉ∈F (\langle\omega^ĉ_{b(∇), U\rangle * \Sigma_{c1̂∈F (\langle\omega^c1̂_{ĉ(∇), V\rangle * X_{c1}̂))}}}}\right) - \Sigma_{ĉ∈F (\langle\omega^ĉ_{b(∇), [U,V]_{VF\rangle * X_ĉ)}}} \rangle \to \langle e^a,\, \Sigma_{ĉ∈F (U(\langle\omega^ĉ_{b(∇), V\rangle) * X_ĉ)}} + \Sigma_{ĉ∈F (\langle\omega^ĉ_{b(∇), V\rangle * \Sigma_{c1̂∈F (\langle\omega^c1̂_{ĉ(∇), U\rangle * X_{c1}̂))}}}} \rangle + \langle e^a,\, -\left(\Sigma_{ĉ∈F (V(\langle\omega^ĉ_{b(∇), U\rangle) * X_ĉ)}} + \Sigma_{ĉ∈F (\langle\omega^ĉ_{b(∇), U\rangle * \Sigma_{c1̂∈F (\lang

In [24]:
for i, step in enumerate(chain_II.steps, 1):
    tag = f"[{step.provenance_tag}]" if step.provenance_tag else ""
    print(f"[{i}] {step.rule} {tag}")
    print(f"    {step.before}")
    print(f" -> {step.after}")
    print()

[1] R form definition [∇,F] [axiom]
    R^a_b(∇)(U, V)
 -> ⟨e^a, R(∇)(U,V)(X_b)⟩

[2] R(∇)(X,Y)Z = ∇_X ∇_Y Z − ∇_Y ∇_X Z − ∇_[X,Y]_VF Z [∇] [axiom]
    R(∇)(U,V)(X_b)
 -> (∇_U(∇_V(X_b)) + (-∇_V(∇_U(X_b))) + (-∇_[U,V]_VF(X_b)))

[3] ∇_V X_b → Σ_c ω^c_b(V)·X_c [∇,F] [axiom]
    ∇_V(X_b)
 -> Σ_ĉ∈F (⟨ω^ĉ_b(∇), V⟩ * X_ĉ)

[4] ∇_X Σ_d body → Σ_d ∇_X body [∇] [axiom]
    ∇_U(Σ_ĉ∈F (⟨ω^ĉ_b(∇), V⟩ * X_ĉ))
 -> Σ_ĉ∈F ∇_U((⟨ω^ĉ_b(∇), V⟩ * X_ĉ))

[5] ∇_X Y-Leibniz [∇] [axiom]
    ∇_U((⟨ω^ĉ_b(∇), V⟩ * X_ĉ))
 -> ((U(⟨ω^ĉ_b(∇), V⟩) * X_ĉ) + (⟨ω^ĉ_b(∇), V⟩ * ∇_U(X_ĉ)))

[6] ∇_V X_b → Σ_c ω^c_b(V)·X_c [∇,F] [axiom]
    ∇_U(X_ĉ)
 -> Σ_c1̂∈F (⟨ω^c1̂_ĉ(∇), U⟩ * X_c1̂)

[7] Σ over Sum: distribute [axiom]
    Σ_ĉ∈F ((U(⟨ω^ĉ_b(∇), V⟩) * X_ĉ) + (⟨ω^ĉ_b(∇), V⟩ * Σ_c1̂∈F (⟨ω^c1̂_ĉ(∇), U⟩ * X_c1̂)))
 -> (Σ_ĉ∈F (U(⟨ω^ĉ_b(∇), V⟩) * X_ĉ) + Σ_ĉ∈F (⟨ω^ĉ_b(∇), V⟩ * Σ_c1̂∈F (⟨ω^c1̂_ĉ(∇), U⟩ * X_c1̂)))

[8] ∇_V X_b → Σ_c ω^c_b(V)·X_c [∇,F] [axiom]
    ∇_U(X_b)
 -> Σ_ĉ∈F (⟨ω^ĉ_b(∇), U⟩ * X_ĉ

## Summary

- **Part A**: $\omega$, $Q$, 1-form (each with $V$-scalar-linearity
  and $V$-additivity); $T$, $R$, 2-form ($C^\infty$-linearity in both
  arguments plus antisymmetry). 14 syntactic closures total are
  available on `CartanFormPropertyProblem`; this notebook shows 4
  representative proofs.
- **Part B.1 / Cartan I**: $T^a = de^a + \omega^a{}_b \wedge e^b$,
  the `CartanStructureProblem` 22-rule bundle closes to $0$ in a single
  `simplify` iteration.
- **Part B.2 / Cartan II**: $R^a{}_b = d\omega^a{}_b + \omega^a{}_c \wedge \omega^c{}_b$,
  the same wrapper plus `CurvatureFormDefinition` +
  `CurvatureDefinitionDefinition` (24 rules total); also closes in a
  single iteration.

The heart of the closure:

1. **Form definitions** ($T^a$, $R^a{}_b$, $\omega^a{}_b$) open the
   atoms into the $\nabla$ + `Pairing` universe.
2. **`ConnectionEvalYFrameDecomposition`** expands $V$ into the frame
   as $\sum_c P(e^c, V)\,X_c$; **`ConnectionFormDecomposition`**
   rewrites $\nabla_V X_b$ as $\sum_c \omega^c{}_b(V)\,X_c$.
3. **`IndexedSum` push-in / Kronecker-contract** rules push the sum
   binder to its destination on the right side and eliminate
   $\delta$-factors.
4. **Cartan-Koszul** `ExteriorDIntrinsicDefinition` reduces the $de^a$
   and $d\omega^a{}_b$ sides into $U(\cdot)$, $V(\cdot)$,
   $[U,V]_{VF}$ components.
5. **Wedge alternation + arity-1 `MultiEval` → `Pairing` bridge**
   expands the right-side $(\alpha \wedge \beta)(U,V)$ expressions as
   $P(\alpha,U)\,P(\beta,V) - P(\alpha,V)\,P(\beta,U)$.
6. Finally `simplify` (notably `sort_product`) brings the `Product`
   factors of both sides into a common canonical order; `collect_terms`
   cancels the sign-paired terms and the residue drops to $0$.